# CSV Ingestion Demo

## Flow

CSV file → Load in Python → Display first 5 rows → Clean column names → Remove duplicates → Check missing values → Save staging output → Save clean output → Generate ingestion log

## 1. Import Required Libraries

In [1]:
import pandas as pd
import json
import uuid
import re
from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    """
    Find project root by walking upward until the data/ folder is found.
    This makes the notebook work even if it is executed from notebooks/data_team/.
    """
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Please make sure a 'data/' folder exists in the project."
    )


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

input_path = PROJECT_ROOT / "data" / "sample_inputs" / "sales.csv"

raw_dir = PROJECT_ROOT / "data" / "raw" / "csv"
staging_dir = PROJECT_ROOT / "data" / "staging" / "csv"
clean_dir = PROJECT_ROOT / "data" / "clean" / "csv"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "sample_raw.csv"
staging_output_path = staging_dir / "sample_staging.csv"
clean_output_path = clean_dir / "sample_clean.csv"
log_output_path = log_dir / "csv_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\duy\notebooks\data_team
Project root: F:\data\new\quanskill\duy
Input path: F:\data\new\quanskill\duy\data\sample_inputs\sales.csv
Input exists: True
Raw output: F:\data\new\quanskill\duy\data\raw\csv\sample_raw.csv
Staging output: F:\data\new\quanskill\duy\data\staging\csv\sample_staging.csv
Clean output: F:\data\new\quanskill\duy\data\clean\csv\sample_clean.csv
Log output: F:\data\new\quanskill\duy\logs\csv_ingestion_log.json


## 3. Validate Input File

In [3]:
if not input_path.exists():
    raise FileNotFoundError(f"Input CSV file not found: {input_path}")

if input_path.stat().st_size == 0:
    raise ValueError(f"Input CSV file is empty: {input_path}")

print("Input file validation passed.")

Input file validation passed.


## 4. Helper Functions

In [4]:
def clean_column_name(column_name: str) -> str:
    """
    Convert column names to clean snake_case format.

    Examples:
    - ORDERNUMBER -> ordernumber
    - Order Number -> order_number
    - ORDER-DATE -> order_date
    """
    column_name = column_name.strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name


def read_csv_with_fallback_encoding(file_path: Path) -> pd.DataFrame:
    """
    Read CSV with fallback encodings.
    This handles common encoding issues in CSV files.
    """
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]

    last_error = None

    for encoding in encodings:
        try:
            print(f"Trying encoding: {encoding}")
            return pd.read_csv(file_path, encoding=encoding)
        except UnicodeDecodeError as error:
            last_error = error
            print(f"Failed encoding: {encoding}")

    raise ValueError(f"Unable to read file with supported encodings. Last error: {last_error}")

## 5. Start Ingestion Run

In [5]:
run_id = str(uuid.uuid4())
source_name = "sales_csv"
source_type = "csv"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: e31a4f32-3dfe-4ea2-94e7-473721f0763e
Start time: 2026-05-29T07:09:43.958666+00:00


## 6. Load Sample CSV

In [6]:
try:
    df_raw = read_csv_with_fallback_encoding(input_path)
    status = "success"
    error_message = None
except Exception as error:
    df_raw = pd.DataFrame()
    status = "failed"
    error_message = str(error)
    raise

records_read = len(df_raw)

print("CSV loaded successfully.")
print("Records read:", records_read)
print("Columns:", df_raw.columns.tolist())

Trying encoding: utf-8
Failed encoding: utf-8
Trying encoding: utf-8-sig
Failed encoding: utf-8-sig
Trying encoding: latin1
CSV loaded successfully.
Records read: 2823
Columns: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']


## 7. Display First 5 Rows

In [7]:
df_raw.head(5)

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 8. Save Raw Copy

In [8]:
df_raw.to_csv(raw_output_path, index=False, encoding="utf-8")
print("Raw copy saved to:", raw_output_path)

Raw copy saved to: F:\data\new\quanskill\duy\data\raw\csv\sample_raw.csv


## 9. Clean Column Names

In [9]:
df_staging = df_raw.copy()

original_columns = df_staging.columns.tolist()
cleaned_columns = [clean_column_name(col) for col in original_columns]

df_staging.columns = cleaned_columns

print("Original columns:")
print(original_columns)

print("\nCleaned columns:")
print(cleaned_columns)

df_staging.head()

Original columns:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Cleaned columns:
['ordernumber', 'quantityordered', 'priceeach', 'orderlinenumber', 'sales', 'orderdate', 'status', 'qtr_id', 'month_id', 'year_id', 'productline', 'msrp', 'productcode', 'customername', 'phone', 'addressline1', 'addressline2', 'city', 'state', 'postalcode', 'country', 'territory', 'contactlastname', 'contactfirstname', 'dealsize']


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 10. Save Parsed Output to Staging

In [10]:
df_staging.to_csv(staging_output_path, index=False, encoding="utf-8")
print("Staging output saved to:", staging_output_path)

Staging output saved to: F:\data\new\quanskill\duy\data\staging\csv\sample_staging.csv


## 11. Check Duplicate Rows

In [11]:
duplicate_count = int(df_staging.duplicated().sum())
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 12. Check Missing Values

In [12]:
missing_values = df_staging.isna().sum()
total_missing_values = int(missing_values.sum())

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values:", total_missing_values)

Missing values by column:
ordernumber            0
quantityordered        0
priceeach              0
orderlinenumber        0
sales                  0
orderdate              0
status                 0
qtr_id                 0
month_id               0
year_id                0
productline            0
msrp                   0
productcode            0
customername           0
phone                  0
addressline1           0
addressline2        2521
city                   0
state               1486
postalcode            76
country                0
territory           1074
contactlastname        0
contactfirstname       0
dealsize               0
dtype: int64

Total missing values: 5157


## 13. Remove Duplicate Rows

In [13]:
df_clean = df_staging.drop_duplicates().copy()

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid after duplicate removal:", records_valid)
print("Records invalid / removed:", records_invalid)

df_clean.head()

Records read: 2823
Records valid after duplicate removal: 2823
Records invalid / removed: 0


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 14. Save Cleaned Output

In [14]:
df_clean.to_csv(clean_output_path, index=False, encoding="utf-8")
print("Clean output saved to:", clean_output_path)

Clean output saved to: F:\data\new\quanskill\duy\data\clean\csv\sample_clean.csv


## 15. Generate Ingestion Log

In [15]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": str(input_path),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "missing_values": missing_values.astype(int).to_dict(),
    "total_missing_values": total_missing_values,
    "error_message": error_message,
    "raw_output_path": str(raw_output_path),
    "staging_output_path": str(staging_output_path),
    "clean_output_path": str(clean_output_path),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Ingestion log saved to:", log_output_path)
ingestion_log

Ingestion log saved to: F:\data\new\quanskill\duy\logs\csv_ingestion_log.json


{'run_id': 'e31a4f32-3dfe-4ea2-94e7-473721f0763e',
 'source_name': 'sales_csv',
 'source_type': 'csv',
 'input_path_or_url': 'F:\\data\\new\\quanskill\\duy\\data\\sample_inputs\\sales.csv',
 'start_time': '2026-05-29T07:09:43.958666+00:00',
 'end_time': '2026-05-29T07:10:08.489349+00:00',
 'status': 'success',
 'records_read': 2823,
 'records_valid': 2823,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'missing_values': {'ordernumber': 0,
  'quantityordered': 0,
  'priceeach': 0,
  'orderlinenumber': 0,
  'sales': 0,
  'orderdate': 0,
  'status': 0,
  'qtr_id': 0,
  'month_id': 0,
  'year_id': 0,
  'productline': 0,
  'msrp': 0,
  'productcode': 0,
  'customername': 0,
  'phone': 0,
  'addressline1': 0,
  'addressline2': 2521,
  'city': 0,
  'state': 1486,
  'postalcode': 76,
  'country': 0,
  'territory': 1074,
  'contactlastname': 0,
  'contactfirstname': 0,
  'dealsize': 0},
 'total_missing_values': 5157,
 'error_message': None,
 'raw_output_path': 'F:\\data\\new\\quanskill\\

## 16. Final Output Check

In [16]:
print("Expected outputs:")

print("Raw output exists:", raw_output_path.exists())
print("Staging output exists:", staging_output_path.exists())
print("Clean output exists:", clean_output_path.exists())
print("Log output exists:", log_output_path.exists())

print("\nOutput paths:")
print("Raw:", raw_output_path)
print("Staging:", staging_output_path)
print("Clean:", clean_output_path)
print("Log:", log_output_path)

Expected outputs:
Raw output exists: True
Staging output exists: True
Clean output exists: True
Log output exists: True

Output paths:
Raw: F:\data\new\quanskill\duy\data\raw\csv\sample_raw.csv
Staging: F:\data\new\quanskill\duy\data\staging\csv\sample_staging.csv
Clean: F:\data\new\quanskill\duy\data\clean\csv\sample_clean.csv
Log: F:\data\new\quanskill\duy\logs\csv_ingestion_log.json


## 17. Summary

Expected generated files:

```text
data/raw/csv/sample_raw.csv
data/staging/csv/sample_staging.csv
data/clean/csv/sample_clean.csv
logs/csv_ingestion_log.json
```